# 03 — Modelos de anomalia (Fase 4)
**Isolation Forest** e **Autoencoder**, comparados aos supervisionados nos cenários **B, C e D**.

Decisões do projeto:
- Treino **só com transações legítimas** (os rótulos servem apenas para filtrar o treino).
- Escopo **só TRANSFER e CASH_OUT**: as demais transações recebem o menor score (aprovação automática por regra).
- A avaliação é na **validação completa**, na mesma base dos supervisionados, para a comparação ser justa.

Runtime: **GPU (T4)** acelera o Autoencoder. Pré-requisito: ter rodado o notebook 02 (usa os scores salvos em `reports/scores`).

In [ ]:
import os, sys, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fraud-detection-paysim'
except ImportError:
    PROJECT_DIR = os.getcwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

import numpy as np, pandas as pd, joblib
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from fraud.config import load_config
cfg = load_config()
y = cfg['data']['target']
FIG = os.path.join(cfg['paths']['reports_dir'], 'figures'); os.makedirs(FIG, exist_ok=True)
MODELS = cfg['paths']['models_dir']; os.makedirs(MODELS, exist_ok=True)
SCORES = os.path.join(cfg['paths']['reports_dir'], 'scores'); os.makedirs(SCORES, exist_ok=True)
pd.options.display.float_format = '{:,.4f}'.format
print('Projeto:', PROJECT_DIR)

## 1. Dados, split e escopo

In [ ]:
from fraud.data import load_parquet
from fraud.split import make_split
from fraud.anomaly import in_scope, AnomalyPrep, fill_out_of_scope, build_autoencoder, reconstruction_error

df = load_parquet(cfg['paths']['parquet'])
train, val, test = make_split(df, cfg)

tr_legit = train[in_scope(train) & (train[y] == 0)]
tr_legit = tr_legit.sample(frac=cfg['train']['legit_frac'], random_state=cfg['seed'])
mask_val = in_scope(val)
val_in = val[mask_val]
print(f"treino (só legítimas no escopo): {len(tr_legit):,} linhas")
print(f"validação: {len(val):,} linhas | no escopo: {mask_val.sum():,} | fraudes: {int(val[y].sum())} "
      f"(todas no escopo: {int(val_in[y].sum()) == int(val[y].sum())})")

## 2. Isolation Forest (cenários B, C e D)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve
from fraud.metrics import evaluate

from fraud.features import SCENARIOS
CENARIOS = {k: SCENARIOS[k] for k in ('B', 'C', 'D')}

resultados, curvas, objetos = [], {}, {}
for letra, variante in CENARIOS.items():
    prep = AnomalyPrep(variante).fit(tr_legit)
    Xtr, Xva = prep.transform(tr_legit), prep.transform(val_in)
    t0 = time.time()
    iso = IsolationForest(n_estimators=200, max_samples=1024, contamination='auto',
                          n_jobs=-1, random_state=cfg['seed']).fit(Xtr)
    s = fill_out_of_scope(-iso.score_samples(Xva), mask_val)
    seg = time.time() - t0
    r = evaluate(val[y].values, s, amount=val['amount'].values)
    r.update({'cenario': letra, 'modelo': 'isolation_forest', 'treino_seg': round(seg, 1)})
    resultados.append(r); curvas[(letra, 'isolation_forest')] = precision_recall_curve(val[y].values, s)[:2]
    np.save(os.path.join(SCORES, f'val_isolation_forest_{letra}.npy'), s.astype('float32'))
    objetos[(letra, 'isolation_forest')] = (prep, iso)
    print(f"ok  cenário {letra} | isolation_forest | {seg:6.1f}s | PR-AUC {r['pr_auc']:.4f}")

## 3. Autoencoder (cenários B, C e D)
O score é o erro de reconstrução: transações que o modelo reconstrói mal são as mais anômalas.

In [ ]:
import keras
EPOCAS = 15
for letra, variante in CENARIOS.items():
    prep = AnomalyPrep(variante).fit(tr_legit)
    Xtr, Xva = prep.transform(tr_legit), prep.transform(val_in)
    ae = build_autoencoder(Xtr.shape[1], seed=cfg['seed'])
    t0 = time.time()
    ae.fit(Xtr, Xtr, epochs=EPOCAS, batch_size=2048, validation_split=0.1, verbose=0,
           callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])
    s = fill_out_of_scope(reconstruction_error(ae, Xva), mask_val)
    seg = time.time() - t0
    r = evaluate(val[y].values, s, amount=val['amount'].values)
    r.update({'cenario': letra, 'modelo': 'autoencoder', 'treino_seg': round(seg, 1)})
    resultados.append(r); curvas[(letra, 'autoencoder')] = precision_recall_curve(val[y].values, s)[:2]
    np.save(os.path.join(SCORES, f'val_autoencoder_{letra}.npy'), s.astype('float32'))
    objetos[(letra, 'autoencoder')] = (prep, ae)
    print(f"ok  cenário {letra} | autoencoder     | {seg:6.1f}s | PR-AUC {r['pr_auc']:.4f}")

res_anom = pd.DataFrame(resultados).set_index(['cenario', 'modelo'])
res_anom.to_csv(os.path.join(cfg['paths']['reports_dir'], 'anomalia_validacao.csv'))

## 4. Comparação com os supervisionados

In [ ]:
sup = pd.read_csv(os.path.join(cfg['paths']['reports_dir'], 'supervisionados_validacao.csv'),
                  index_col=['cenario', 'modelo'])
sup = sup[sup.index.get_level_values('cenario').isin(['B', 'C', 'D'])]
comp = pd.concat([sup, res_anom[sup.columns.intersection(res_anom.columns)]]).sort_index()
ordem = ['logreg', 'random_forest', 'xgboost', 'isolation_forest', 'autoencoder']
for nome, col, f in [('PR-AUC (validação)', 'pr_auc', None),
                     ('Recall em VALOR no corte de melhor F1', 'bestF1_value_recall', None),
                     ('Taxa de falsos positivos no corte de melhor F1', 'bestF1_fpr', None)]:
    print(nome)
    display(comp[col].unstack('modelo')[ordem])
comp.to_csv(os.path.join(cfg['paths']['reports_dir'], 'comparacao_5_modelos_validacao.csv'))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, letra in zip(axes, CENARIOS):
    for nome in ['random_forest', 'xgboost']:
        s = np.load(os.path.join(SCORES, f'val_{nome}_{letra}.npy'))
        p, r = precision_recall_curve(val[y].values, s)[:2]
        ax.plot(r, p, label=nome, alpha=0.8)
    for nome in ['isolation_forest', 'autoencoder']:
        p, r = curvas[(letra, nome)]
        ax.plot(r, p, label=nome, linestyle='--')
    ax.set_title(f'Cenário {letra} — {CENARIOS[letra]}', fontsize=10)
    ax.set_xlabel('recall'); ax.legend(fontsize=8)
axes[0].set_ylabel('precision')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'anomalia_pr_curves.png'), dpi=130); plt.show()

In [ ]:
# Salva os modelos de anomalia do cenário B (principal) para as próximas fases
iso_prep, iso = objetos[('B', 'isolation_forest')]
joblib.dump({'prep': iso_prep, 'model': iso}, os.path.join(MODELS, 'isolation_forest_B.joblib'), compress=3)
ae_prep, ae = objetos[('B', 'autoencoder')]
joblib.dump(ae_prep, os.path.join(MODELS, 'autoencoder_B_prep.joblib'), compress=3)
ae.save(os.path.join(MODELS, 'autoencoder_B.keras'))
print('modelos salvos em', MODELS)

## 5. Imagem com os resultados (para enviar de volta)

In [ ]:
# Gera uma imagem com as tabelas e baixa (também fica salva em reports/figures)
def tabela(ax, df, titulo, fmt):
    ax.axis('off')
    ax.set_title(titulo, fontsize=11, fontweight='bold', loc='left')
    texto = [[fmt.format(v) for v in linha] for linha in df.values]
    t = ax.table(cellText=texto, rowLabels=[f'Cenário {i}' for i in df.index],
                 colLabels=list(df.columns), loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1, 1.5)

fig, axes = plt.subplots(3, 1, figsize=(10, 8.5))
tabela(axes[0], comp['pr_auc'].unstack('modelo'), 'PR-AUC (validação)', '{:.4f}')
tabela(axes[1], comp['bestF1_value_recall'].unstack('modelo'), 'Recall em valor (corte de melhor F1)', '{:.1%}')
tabela(axes[2], comp['bestF1_fpr'].unstack('modelo'), 'Taxa de falsos positivos (corte de melhor F1)', '{:.4%}')
fig.suptitle('Fase 4 — os 5 modelos, validação (split por linhas)', fontsize=12)
fig.tight_layout()
img1 = os.path.join(FIG, 'resultado_5_modelos.png')
fig.savefig(img1, dpi=150, bbox_inches='tight'); plt.show()
try:
    from google.colab import files
    for arq in [img1, os.path.join(FIG, 'anomalia_pr_curves.png')]:
        files.download(arq)
except ImportError:
    print('Imagens salvas em', FIG)

## O que enviar de volta
As linhas `ok ...` das seções 2 e 3, as imagens baixadas na seção 5 e a mensagem `modelos salvos em ...`.